# One sentence to a Hertzsprung–Russell diagram

This demo gives Augplot a stellar table with colour, magnitude, and class labels, plus one short plotting request. Inspect the generated figure and Python to judge the one-shot result.

**Setup:** Install Augplot as in the [quick start](../README.md#quick-start), select that Python kernel, and have an OpenAI API key ready. The catalog download and model call need internet access. Generated Python runs locally after validation, without an OS sandbox.

In [ ]:
import os
from getpass import getpass

import pandas as pd

import augplot as ap

os.environ["AUGPLOT_MODEL"] = "openai/gpt-6-sol"
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

## Load an open stellar catalog

The [HYG Database v4.1](https://github.com/astronexus/HYG-Database/tree/main/hyg/CURRENT) by astronexus is licensed [CC BY-SA 4.0](https://github.com/astronexus/HYG-Database/blob/main/LICENSE). Its ci field is B−V colour index, absmag is absolute visual magnitude, and spect contains catalog spectral labels. The notebook downloads the catalog directly; no data file is bundled.

In [ ]:
catalog_url = (
    "https://raw.githubusercontent.com/astronexus/HYG-Database/main/"
    "hyg/CURRENT/hygdata_v41.csv"
)
catalog = pd.read_csv(
    catalog_url, usecols=["id", "dist", "mag", "absmag", "ci", "spect"]
)
print(f"Downloaded {len(catalog):,} catalog rows")

## Prepare the plotting columns and class labels

Exclude the catalog's distance sentinel (≥ 100,000 pc, meaning missing or dubious parallax). Combine bright stars (apparent magnitude ≤ 6.5) with nearby stars (distance ≤ 75 pc), then use a stated display range of B−V from −0.4 to 2.4 mag and absolute V magnitude from −8 to 17 mag. Parse only unambiguous spectral labels into luminosity classes; all other stars remain Unclassified. Any class guide curves should summarize the labeled catalog stars, not imply exact physical boundaries. This is a visual demonstration sample, not a population-complete survey. The cell reports how many selected rows fall outside the range.

In [ ]:
usable = catalog.dropna(subset=["dist", "mag", "absmag", "ci"])
usable = usable[usable["dist"] < 100000]
selected = usable[(usable["mag"] <= 6.5) | (usable["dist"] <= 75)]
in_range = selected[
    selected["ci"].between(-0.4, 2.4)
    & selected["absmag"].between(-8, 17)
]
spectral_type = in_range["spect"].fillna("").str.upper().str.strip()
class_patterns = {
    "White dwarfs": r"^D[A-Z0-9.]*$",
    "Supergiants": r"^[OBAFGKM][0-9.]*I(?:A|B|AB)?$",
    "Bright giants": r"^[OBAFGKM][0-9.]*II$",
    "Giants": r"^[OBAFGKM][0-9.]*III$",
    "Subgiants": r"^[OBAFGKM][0-9.]*IV$",
    "Main sequence": r"^[OBAFGKM][0-9.]*V$",
}
stellar_class = pd.Series("Unclassified", index=in_range.index)
for label, pattern in class_patterns.items():
    stellar_class.loc[spectral_type.str.fullmatch(pattern)] = label
stars = (
    in_range.rename(
        columns={
            "ci": "B_minus_V_color_index_mag",
            "absmag": "absolute_V_magnitude_mag",
        }
    )[["B_minus_V_color_index_mag", "absolute_V_magnitude_mag"]]
    .assign(stellar_class=stellar_class)
    .reset_index(drop=True)
)
print(f"Selected {len(selected):,} stars; plotting {len(stars):,}")
print(f"Outside the stated display range: {len(selected) - len(stars):,}")
print(stars["stellar_class"].value_counts().to_string())
stars.head()

## Ask once

The plotting prompt is a single sentence. It requests the labeled category curves shown in classic H–R illustrations, using the catalog's class labels as evidence. Augplot opens a notebook input field only if the model asks for an essential choice; otherwise this makes one plotting request.

In [ ]:
request = (
    "Make an H–R diagram of all stars with data-derived guide curves for main sequence, "
    "subgiants, giants, bright giants, supergiants, and white dwarfs."
)
chart = ap.plot(stars, prompt=request, backend="matplotlib")

## Inspect the one-shot result

A good observational H–R diagram should put B−V colour on the horizontal axis, place brighter absolute magnitudes higher, and show the stellar sequence. The curves should summarize catalog class labels and read as guides, not exact boundaries. Review the figure and generated function below; the notebook does not substitute a hand-written plot.

In [ ]:
from IPython.display import Code, display

print(chart.explanation)
print("Reused cached code:", chart.cache_hit)
display(Code(chart.code, language="python"))